# Transcript Corruption Engine (Error-Propagation Study, proposal 4.6)

Wraps `src/error_propagation/corrupt_transcripts_ven.py` - corrupts clean transcripts to a target
WER, for the error-propagation experiment: run the misinformation classifier
on transcripts corrupted at 10/20/30/40/50% WER and see how much accuracy
degrades (Objectives 5-6). This notebook demonstrates and verifies the
engine; the actual degradation-curve experiment runs once a trained
classifier exists (see `train_classifier.ipynb`).

## 1. Corrupt the NCHLT test set at each target WER

In [ ]:
import sys
sys.path.insert(0, "../../src/error_propagation")
from corrupt_transcripts_ven import corrupt_file

levels = [0.10, 0.20, 0.30, 0.40, 0.50]
for level in levels:
    corrupt_file(
        input_csv="../dataset/processed/nchlt_ven/test.csv",
        output_csv=f"../results/corrupted/nchlt_test_wer{int(level*100)}.csv",
        target_wer=level, mode="mixed", seed=42,
    )

## 2. Verify achieved WER matches target

In [ ]:
import csv, jiwer

for level in levels:
    rows = list(csv.DictReader(open(f"../results/corrupted/nchlt_test_wer{int(level*100)}.csv")))
    refs = [r["transcript_clean"] for r in rows]
    hyps = [r["transcript"] for r in rows]
    achieved = jiwer.wer(refs, hyps)
    print(f"target {level:.2f} -> achieved {achieved:.3f}")

## 3. Using a real measured error model (once ASR predictions exist)

In [ ]:
# from corrupt_transcripts_ven import ErrorModel
# em = ErrorModel.from_prediction_files([
#     "../results/preds_pilot_v2/wav2vec2-final_nchlt_test.csv",
# ])
# print("measured S:D:I ratio:", em.ratio)
# corrupt_file(..., error_model=em)  # pass into corrupt_sentence via corrupt_file's --error-model equivalent